# ETL MV Agent-Only Flow

本 notebook 用于第一轮 Agent-only 原型联调：依次运行 `SQLLoaderAgent`、`FeatureAgent`、`FamilyAgent` 和 `SelfIterationAgent`。

当前 notebook 只验证执行前分析链路和自迭代反馈链路，不包含 batch clustering、MV generation、rewrite 或 executor。

In [1]:
from pathlib import Path
import sys
from dotenv import load_dotenv

# 定位项目根目录，保证从 notebook 子目录运行时也能导入 llm_demo 包。
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != 'MV_gen' and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

# 将项目根目录加入 Python import path，避免依赖 notebook 的启动目录。
sys.path.insert(0, str(PROJECT_ROOT))

# 只从项目根目录 .env 读取 DeepSeek 配置，不在 notebook 中硬编码 key。
load_dotenv(PROJECT_ROOT / '.env')

True

In [2]:
from datetime import datetime
from llm_demo.src.core.artifact_store import ArtifactStore
from llm_demo.src.core.llm_client import LLMClient
from llm_demo.src.agents.sql_loader_agent import SQLLoaderAgent
from llm_demo.src.agents.feature_agent import FeatureAgent
from llm_demo.src.agents.family_agent import FamilyAgent
from llm_demo.src.agents.self_iteration_agent import SelfIterationAgent

# 每次实验使用一个独立 run_id，所有 Artifact 都写入 llm_demo/artifacts/{run_id}/。
run_id = datetime.now().strftime('%Y%m%d_%H%M%S')
store = ArtifactStore(run_id=run_id, artifact_root=PROJECT_ROOT / 'llm_demo' / 'artifacts')

# LLMClient 会从 .env 读取 DeepSeek API 配置，并提供 JSON 输出解析能力。
llm_client = LLMClient(project_root=PROJECT_ROOT)

In [4]:
# 第一轮 tracer bullet 使用 q42/q52 验证 QueryBlock 提取和 QueryFamily 聚合。
sql_paths = [
    PROJECT_ROOT / 'tpcds-spark' / 'q42.sql',
    PROJECT_ROOT / 'tpcds-spark' / 'q52.sql',
]

# 1. 读取原始 SQL，并落盘到 00_raw_sql/。
raw_sql_dir = SQLLoaderAgent(store).run(sql_paths)

# 2. 调用 LLM + rules，从 SQL Text 提取 QueryBlock、query_to_qbs 和 qb_to_query。
query_blocks_path = FeatureAgent(store, llm_client).run(raw_sql_dir)

# 3. 基于 QueryBlock 的 family_key / join skeleton 聚合 QueryFamily。
families_path = FamilyAgent(store, llm_client).run(query_blocks_path)

# 4. 读取 run log，生成带证据引用的自迭代反馈建议。
feedback_path = SelfIterationAgent(store, llm_client).run(store.run_log_path)

RuntimeError: LLM inference failed: Request timed out.

In [ ]:
# 查看本次 run 的核心 Artifact 路径，便于人工检查 LLM 输出是否合理。
print('run_id:', run_id)
print('query_blocks:', query_blocks_path)
print('families:', families_path)
print('feedback:', feedback_path)
print('run_log:', store.run_log_path)